# Newman/Ratner quality pilot on Colab

Choose a GPU runtime in Colab or through the official Google Colab extension for VS Code. This notebook tests one 9.95-minute interview, `5694MC`, with mother, experimenter, and child turns. It runs CHAT-guided word alignment and prosody, plus independent Silero-VAD ASR and pyannote diarization.

Before running, accept the access conditions for [pyannote/speaker-diarization-community-1](https://huggingface.co/pyannote/speaker-diarization-community-1). Enter your Hugging Face read token in the password widget below and click **Set token** before continuing. It is held only in this runtime's environment and is never placed in a code cell or printed.

Upload these two files from the local `data/staging/newman_ratner` tree to the runtime's `/content` directory: `media/Interviews/18/5694MC.mp3` and `transcripts/extracted/Interviews/18/5694MC.cha`. In VS Code, right-click each local file in Explorer and choose **Upload to Colab**; in the Colab browser, use the Files sidebar. The repository excludes the data tree. The model and package downloads need internet access. Results in `/content` disappear when the Colab runtime ends. In VS Code, use the Colab **Contents** view to right-click the generated ZIP and choose **Download**; in the browser, use the Files sidebar.


In [ ]:
from pathlib import Path
import os, subprocess, shutil, sys
repo = Path('/content/ytspeechpipeline')
if not (repo / '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'codex/newman-quality-pilot',
                    'https://github.com/vladinsc/ytspeechpipeline.git', str(repo)], check=True)
os.chdir(repo)
if shutil.which('ffmpeg') is None or shutil.which('ffprobe') is None:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=True)
print('Repository:', repo)


In [ ]:
%pip install -q 'whisperx>=3.8.4,<3.9' 'praat-parselmouth>=0.4.3' 'numpy>=2.1,<3' ipywidgets


In [ ]:
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime, then reconnect.'
print(torch.cuda.get_device_name(0), 'VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
import ipywidgets as widgets
from IPython.display import display
password = widgets.Password(description='HF_TOKEN')
set_button = widgets.Button(description='Set token')
status = widgets.Output()
def set_token(_):
    if not password.value.strip():
        return
    os.environ['HF_TOKEN'] = password.value.strip()
    password.value = ''
    password.disabled = True
    set_button.disabled = True
    with status:
        print('Token set in this runtime (value hidden).')
set_button.on_click(set_token)
display(password, set_button, status)


In [ ]:
assert os.environ.get('HF_TOKEN'), 'Enter the token in the widget and click Set token first.'
# Confirm both files were uploaded to /content before running this cell.
relative_paths = {
    '5694MC.mp3': 'media/Interviews/18/5694MC.mp3',
    '5694MC.cha': 'transcripts/extracted/Interviews/18/5694MC.cha',
}
for name, relative in relative_paths.items():
    source = Path('/content') / name
    if not source.is_file():
        raise FileNotFoundError(f'Upload {name} to /content and rerun this cell.')
    destination = repo / 'data/staging/newman_ratner' / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)
    print(name, 'uploaded:', round(destination.stat().st_size / 1e6, 2), 'MB')


The first run uses `medium.en`, int8, batch size 1, and four guided turns per speaker. This checks the complete path on limited VRAM. For thesis quality, repeat later with `large-v3` and `float16` in a **new** output directory if the available GPU has enough memory. Model changes cannot resume into the same results directory.


In [ ]:
source_id = 'NewmanRatner:18:5694MC:interview'
out_dir = repo / 'results/newman_quality_colab_medium'
base_command = [sys.executable, 'tools/run_newman_quality_pilot.py',
                '--source-id', source_id, '--out-dir', str(out_dir),
                '--whisper-model', 'medium.en', '--compute-type', 'int8',
                '--batch-size', '1', '--turns-per-speaker', '4']
subprocess.run(base_command + ['--dry-run'], check=True, env=os.environ.copy())
subprocess.run(base_command, check=True, env=os.environ.copy())


In [ ]:
import json
work = out_dir / source_id.replace(':', '_')
report = json.loads((work / 'report.json').read_text(encoding='utf-8'))
print('Speaker coverage:', report['speaker_diarization']['speaker_coverage'])
print('Mapped speaker accuracy on covered speech:', report['speaker_diarization']['mapped_accuracy_on_covered_speech'])
print('Diagnostic ASR WER:', report['independent_asr']['diagnostic_wer'])
print('Aligned word coverage:', report['guided_alignment']['aligned_word_coverage'])
print('Review videos:', report['artifacts'].get('review_videos', []))
archive = shutil.make_archive('/content/newman_quality_colab_medium', 'zip', root_dir=out_dir)
print('Download the results ZIP:', archive)
